<a href="https://colab.research.google.com/github/adarsh-crafts/BSDA5002-Mathematical-Foundations-of-Generative-AI/blob/main/01%20-%20Simple%20Neural%20Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

# Hyperparameters

In [ ]:
batch_size = 64
learning_rate = 1e-3
epochs = 5

# Load FashionMNIST

In [ ]:
# download MNIST data

train_dataset = datasets.FashionMNIST(
    root='data',
    train=True,
    download=True,
    transform = v2.Compose(
        [v2.ToImage(),  # convert to torch image
         v2.ToDtype(torch.float32, scale=True)]   # convert to float32 and scale b/w [0,1]
    )
)

test_dataset = datasets.FashionMNIST(
    root='data',
    train=False,
    download=True,
    transform=v2.Compose(
        [v2.ToImage(),
         v2.ToDtype(torch.float32, scale=True)]
    )
)


# create dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 203kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.74MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 12.3MB/s]


# For a Custom Dataset

We use csv file format: `path, label`

In [ ]:
import os
import pandas as pd
from torchvision.io import read_image
from torch.utils.data import Dataset

# create dataset class
class CustomDataset(Dataset):
  def __init__(self, annot_file, img_dir, transform=None, target_transform=None):
    self.img_labels = pd.read_csv(annot_file)
    self.img_dir = img_dir
    self.transform = transform
    self.target_transform = target_transform

  def __len__(self):
    return len(self.img_labels)

  def __getitem__(self, idx):
    # get image and its label
    img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])   # path in the first column
    image = read_image(img_path)
    label = self.img_labels.iloc[idx, 1]

    # transforms
    if self.transform:
      image = self.transform(image)
    if self.target_tranform:
      label = self.transform(label)

    return image, label


try:
  # create the dataset objects
  train_dataset = CustomDataset(
      annot_file='path_to_train_csv.csv',
      img_dir='path_to_train_data',
      transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
  )

  test_dataset = CustomDataset(
      annot_file='path_to_test_csv.csv',
      img_dir='path_to_test_data',
      transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
  )


  # create dataloaders
  train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

except FileNotFoundError:
  print('No data found in the custom path specified.')

No data found in the custom path specified.


# Build Neural Network

In [ ]:
# check and use GPU or MPS if available
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
print(device)

cuda


In [ ]:
# define NN class
class CustomNN(nn.Module):
  def __init__(self):
    super().__init__()

    # flatten img cos we using simple mlp, not cnn
    self.flatten = nn.Flatten()
    self.hidden_layers = nn.Sequential(
        nn.Linear(28*28, 728),
        nn.ReLU(),
        nn.Linear(728, 728),
        nn.ReLU(),
        nn.Linear(728, 10)
    )

  def forward(self, x):
    x_flattened = self.flatten(x)
    logits = self.hidden_layers(x_flattened)
    return logits

In [ ]:
# create model obj and send it to device
model = CustomNN().to(device)
print(model)

print('\nStructure of Model')
for name, param in model.named_parameters():
  print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

CustomNN(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (hidden_layers): Sequential(
    (0): Linear(in_features=784, out_features=728, bias=True)
    (1): ReLU()
    (2): Linear(in_features=728, out_features=728, bias=True)
    (3): ReLU()
    (4): Linear(in_features=728, out_features=10, bias=True)
  )
)

Structure of Model
Layer: hidden_layers.0.weight | Size: torch.Size([728, 784]) | Values : tensor([[ 0.0050,  0.0176,  0.0091,  ..., -0.0157, -0.0081,  0.0217],
        [ 0.0336,  0.0105, -0.0288,  ...,  0.0252,  0.0130,  0.0162]],
       device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: hidden_layers.0.bias | Size: torch.Size([728]) | Values : tensor([-0.0082, -0.0125], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: hidden_layers.2.weight | Size: torch.Size([728, 728]) | Values : tensor([[ 0.0110,  0.0145, -0.0356,  ...,  0.0113, -0.0350,  0.0183],
        [ 0.0006, -0.0030,  0.0107,  ...,  0.0124,  0.0120, -0.0125]],
       device='cuda:0', grad_fn=<SliceBackward0>) 

Lay

In [ ]:
# loss fn and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
# define train and test loops

def train(model, dataloader, loss_fn, optimizer):
  running_loss, correct_preds, total = 0, 0, 0

  model.train()
  for batch, (X, y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)

    pred = model(X)
    loss = loss_fn(pred, y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    running_loss += loss.item()
    correct_preds += (pred.argmax(1) == y).sum().item()
    total += y.size(0)

  epoch_loss = running_loss / len(dataloader)
  epoch_accuracy = (correct_preds*100)/total

  return epoch_loss, epoch_accuracy


def test(model, dataloader, loss_fn):
  running_loss, correct_preds, total = 0, 0, 0

  model.eval()
  with torch.no_grad():
    for X, y in dataloader:
      X, y = X.to(device), y.to(device)
      pred = model(X)
      loss = loss_fn(pred, y)

      running_loss += loss.item()
      correct_preds += (pred.argmax(1) == y).sum().item()
      total += y.size(0)

  eval_loss = running_loss / len(dataloader)
  eval_accuracy = (correct_preds*100)/total

  return eval_loss, eval_accuracy

In [ ]:
# train model

for e in range(epochs):
  epoch_loss, epoch_acc = train(model, train_dataloader, loss_fn, optimizer)
  eval_loss, eval_acc = test(model, test_dataloader, loss_fn)
  print(f'Epoch: {e+1} | Train {{ loss: {epoch_loss}, acc: {epoch_acc} }} | Test {{ {eval_loss}, acc: {eval_acc} }}')

Epoch: 1 | Train { loss: 0.9918580193763603, acc: 66.84666666666666 } | Test { 0.9562206203770486, acc: 66.28 }
Epoch: 2 | Train { loss: 0.9100874944536417, acc: 68.17666666666666 } | Test { 0.8902352380145128, acc: 67.6 }
Epoch: 3 | Train { loss: 0.8522054849784257, acc: 69.51833333333333 } | Test { 0.8423983067463917, acc: 69.18 }
Epoch: 4 | Train { loss: 0.8087806422064807, acc: 70.79666666666667 } | Test { 0.8050363766159981, acc: 70.98 }
Epoch: 5 | Train { loss: 0.7743616056467678, acc: 72.25666666666666 } | Test { 0.7745584534232024, acc: 71.43 }
